# Generative AI 011 — Runnable Primitives and LCEL

Five primitives connect everything else in LangChain. All five run here with
**no API key**.

| Part | What we check |
|---|---|
| A | sequence returns a value; parallel returns a **dict** |
| B | passthrough keeps what a sequence would consume |
| C | lambda — and why counting belongs there, not in a prompt |
| D | branch — pay for the second call only when needed |
| E | `a \| b \| c` is **identical** to `RunnableSequence(a, b, c)` |
| F | a dict in a pipe becomes `RunnableParallel`; `&` raises |

Needs `langchain-core`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.runnables import (RunnableSequence, RunnableParallel,
                                      RunnablePassthrough, RunnableLambda,
                                      RunnableBranch)

parser = StrOutputParser()
JOKE = "Why did the neural network go to therapy? It had too many layers."

## Part A — Sequence and Parallel

The difference that matters is the **shape** of what comes back.

In [ ]:
seq = RunnableSequence(
    PromptTemplate.from_template("Write a joke about {topic}"),
    FakeListChatModel(responses=["A joke about AI."]),
    parser,
)
print("sequence ->", repr(seq.invoke({"topic": "AI"})))

parallel = RunnableParallel({
    "tweet":    PromptTemplate.from_template("Tweet about {topic}")
                | FakeListChatModel(responses=["AI is here. #AI"]) | parser,
    "linkedin": PromptTemplate.from_template("LinkedIn post about {topic}")
                | FakeListChatModel(responses=["Thoughts on AI..."]) | parser,
})
out = parallel.invoke({"topic": "AI"})
print("parallel ->", out)
print("keys     ->", list(out))

assert isinstance(out, dict) and set(out) == {"tweet", "linkedin"}

A sequence returns whatever its last step returned. A parallel returns a
**dict keyed by branch name** — and those keys are exactly what the next
template's placeholders bind to.

## Part B — RunnablePassthrough

In [ ]:
for value in (2, "hi", {"a": 1}, [1, 2]):
    assert RunnablePassthrough().invoke(value) == value
    print(f"{value!r:<12} -> {RunnablePassthrough().invoke(value)!r}")
print("\nIdentity, for any type.")

In [ ]:
joke_chain = (PromptTemplate.from_template("Write a joke about {topic}")
              | FakeListChatModel(responses=[JOKE]) | parser)
explain = (PromptTemplate.from_template("Explain this joke: {text}")
           | FakeListChatModel(responses=["Because it had layers."]) | parser)

# The problem: a sequence CONSUMES its intermediate values.
print("chained  ->", repr((joke_chain | explain).invoke({"topic": "AI"})))
print("          the joke itself is gone.\n")

# The fix: one branch explains, the other hands the joke along untouched.
both = joke_chain | RunnableParallel({
    "joke": RunnablePassthrough(),
    "explanation": explain,
})
result = both.invoke({"topic": "AI"})
for k, v in result.items():
    print(f"{k:<12}: {v!r}")

assert result["joke"] == JOKE

## Part C — RunnableLambda

Any Python function becomes a component. The interesting question is *what
belongs there*.

In [ ]:
def word_counter(text):
    return len(text.split())

print(RunnableLambda(word_counter).invoke("This is a sample text"))

chain = joke_chain | RunnableParallel({
    "joke": RunnablePassthrough(),
    "word_count": RunnableLambda(lambda x: len(x.split())),
})
result = chain.invoke({"topic": "AI"})
print(result["word_count"], "words")

# Exact, every time - unlike asking the model to count.
assert result["word_count"] == len(JOKE.split()) == 13
print("matches the real count:", len(JOKE.split()))

**Put deterministic work in a lambda, not in a prompt.** Counting is exact
arithmetic and models are poor at it. `len(x.split())` is right every time,
instant, and free.

## Part D — RunnableBranch

In [ ]:
summariser = (PromptTemplate.from_template("Summarise:\n{text}")
              | FakeListChatModel(responses=["A short summary."]) | parser)

THRESHOLD = 50
branch = RunnableBranch(
    (lambda x: len(x.split()) > THRESHOLD, summariser),   # if
    RunnablePassthrough(),                                # else
)

short = "A brief report on the topic."
long_ = " ".join(["word"] * 120)

for label, text in (("short", short), ("long", long_)):
    out = branch.invoke(text)
    took = "summarised" if out != text else "passed through"
    print(f"{label:<6} {len(text.split()):>4} words -> {took}")

assert branch.invoke(short) == short          # untouched
assert branch.invoke(long_) != long_          # summarised

The default at the end is the `else`. Making it a passthrough is what keeps
the short report intact — so **you only pay for the second model call when the
text is actually long**.

## Part E — Is the pipe really the same thing?

In [ ]:
prompt = PromptTemplate.from_template("Write a joke about {topic}")
model  = FakeListChatModel(responses=[JOKE])

explicit = RunnableSequence(prompt, model, parser)
piped    = prompt | model | parser

print("same type  :", type(explicit).__name__ == type(piped).__name__,
      f"({type(piped).__name__})")
print("same steps :", [type(s).__name__ for s in explicit.steps]
                   == [type(s).__name__ for s in piped.steps])
print("same output:", explicit.invoke({"topic": "AI"}) == piped.invoke({"topic": "AI"}))

assert type(explicit).__name__ == type(piped).__name__
assert explicit.invoke({"topic": "AI"}) == piped.invoke({"topic": "AI"})

Identical, three ways. The pipe is shorthand for the same object — not a
different mechanism.

## Part F — Two claims about LCEL that no longer hold

You will often read that LCEL only covers sequences, with speculation about a
future `&` operator for parallel. Check both.

In [ ]:
# A plain DICT in a pipe position:
chain = (prompt | model | parser) | {
    "joke": RunnablePassthrough(),
    "words": RunnableLambda(lambda x: len(x.split())),
}
print("type of that step:", type(chain.steps[-1]).__name__)
print("result           :", chain.invoke({"topic": "AI"}))

assert type(chain.steps[-1]).__name__ == "RunnableParallel"
print("\n-> LCEL DOES cover parallel. You rarely name the class at all.")

In [ ]:
try:
    RunnablePassthrough() & RunnableLambda(lambda x: x)
    print("& works (unexpected)")
except TypeError as e:
    print("&:", e)
    print("\n-> no such operator. The dict form made one unnecessary.")

In [ ]:
# One more shorthand: add a key while keeping everything already there.
add = RunnablePassthrough.assign(words=lambda x: len(x["joke"].split()))
print(add.invoke({"joke": "a b c d"}))
assert add.invoke({"joke": "a b c d"}) == {"joke": "a b c d", "words": 4}

## What to take away

- **Five primitives** connect everything: Sequence, Parallel, Passthrough,
  Lambda, Branch.
- A sequence returns a value; a **parallel returns a dict** whose keys bind to
  the next template.
- **Passthrough** keeps a value a sequence would consume.
- **Lambda** is where deterministic work belongs — the word count was exact.
- **Branch** is if/else; a passthrough default means short inputs cost nothing.
- `a | b | c` is **identical** to `RunnableSequence(a, b, c)`.
- A **dict in a pipe becomes `RunnableParallel`**; `&` raises `TypeError`.

## Exercises

1. Build a chain that returns the joke, its explanation, **and** its word count
   in one dict. How many primitives did you need?
2. `RunnablePassthrough.assign` can replace a parallel-with-a-passthrough-branch
   in some cases. Rewrite Part B with it. Which reads better?
3. Add a second condition to the branch (say, summarise differently above 500
   words). Does order matter among the `(condition, runnable)` pairs?
4. Put a `RunnableLambda` that raises an exception into a chain. Where does the
   error surface, and how much of the chain had already run?
5. Build a chain from a Python list at run time. This is the case where naming
   `RunnableSequence` beats the pipe — why?